In [42]:
from pygom.model.transition import Transition
from pygom.model.pygom_model import Model

In [43]:
living_states = ['S', 'E', 'I', 'R']
dead_states = ['D']
states = living_states + dead_states

params = ['beta', 'alpha', 'gamma', 'mu_b', 'mu_d', 'p']
derived_params = [('N', 'S+E+I+R')]

inf = Transition(origin='S', destination='E', rate='beta*S*I/N')
prog = Transition(origin='E', destination='I', rate='alpha*E')
rec = Transition(origin='I', destination='R', rate='gamma*I*(1-p)')
fatal = Transition(origin='I', destination='D', rate='gamma*I*p')

birth = Transition(destination='S', rate='mu_b*N')
deaths = [Transition(origin=state, rate=f'mu_d*{state}') for state in living_states]

event_list = [inf, prog, rec, fatal, birth] + deaths

mod = Model(state=states, param=params, derived_param=derived_params, event=event_list)

# Maths

In [ ]:
# Set some example parameters for numerical evaluation

state_example = [700, 200, 100, 50, 0]
time_example = 0.0

mod.parameters = {
    'beta': 0.3,
    'alpha': 0.5,
    'gamma': 0.25,
    'mu_b': 1/(100*365),
    'mu_d': 1/(100*365),
    'p': 0.1
}

## Foundations

### State change matrix

In [45]:
mod.state_change_matrix.get_equation()

Matrix([
[-1,  0,  0,  0, 1, -1,  0,  0,  0],
[ 1, -1,  0,  0, 0,  0, -1,  0,  0],
[ 0,  1, -1, -1, 0,  0,  0, -1,  0],
[ 0,  0,  1,  0, 0,  0,  0,  0, -1],
[ 0,  0,  0,  1, 0,  0,  0,  0,  0]])

In [46]:
mod.state_change_matrix(state_example, time_example)

array([[-1.,  0.,  0.,  0.,  1., -1.,  0.,  0.,  0.],
       [ 1., -1.,  0.,  0.,  0.,  0., -1.,  0.,  0.],
       [ 0.,  1., -1., -1.,  0.,  0.,  0., -1.,  0.],
       [ 0.,  0.,  1.,  0.,  0.,  0.,  0.,  0., -1.],
       [ 0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.]])

### ODEs

In [47]:
mod.ode.get_equation()

Matrix([
[                 -I*S*beta/N + N*mu_b - S*mu_d],
[                -E*alpha - E*mu_d + I*S*beta/N],
[E*alpha - I*gamma*p - I*gamma*(1 - p) - I*mu_d],
[                      I*gamma*(1 - p) - R*mu_d],
[                                     I*gamma*p]])

In [48]:
mod.ode(state_example, time_example)

array([[-19.99041096],
       [-80.00547945],
       [ 74.99726027],
       [ 22.49863014],
       [  2.5       ]])

### Event rate vector

In [49]:
mod.event_rate_vector.get_equation()

Matrix([
[     I*S*beta/N],
[        E*alpha],
[I*gamma*(1 - p)],
[      I*gamma*p],
[         N*mu_b],
[         S*mu_d],
[         E*mu_d],
[         I*mu_d],
[         R*mu_d]])

In [50]:
mod.event_rate_vector(state_example, time_example)

array([[2.00000000e+01],
       [1.00000000e+02],
       [2.25000000e+01],
       [2.50000000e+00],
       [2.87671233e-02],
       [1.91780822e-02],
       [5.47945205e-03],
       [2.73972603e-03],
       [1.36986301e-03]])

## Jacobians

### ODE wrt params

In [51]:
mod.ode_jacobian_params.get_equation()

Matrix([
[-I*S/N,  0,                0, N, -S,        0],
[ I*S/N, -E,                0, 0, -E,        0],
[     0,  E, -I*p - I*(1 - p), 0, -I,        0],
[     0,  0,        I*(1 - p), 0, -R, -I*gamma],
[     0,  0,              I*p, 0,  0,  I*gamma]])

In [52]:
mod.ode_jacobian_params(state_example, time_example)

array([[ -66.66666667,    0.        ,    0.        , 1050.        ,
        -700.        ,    0.        ],
       [  66.66666667, -200.        ,    0.        ,    0.        ,
        -200.        ,    0.        ],
       [   0.        ,  200.        , -100.        ,    0.        ,
        -100.        ,    0.        ],
       [   0.        ,    0.        ,   90.        ,    0.        ,
         -50.        ,  -25.        ],
       [   0.        ,    0.        ,   10.        ,    0.        ,
           0.        ,   25.        ]])

### ODE wrt states

In [53]:
mod.ode_jacobian_states.get_equation()

Matrix([
[-I*beta/N + I*S*beta/N**2 + mu_b - mu_d,          I*S*beta/N**2 + mu_b, I*S*beta/N**2 + mu_b - S*beta/N, I*S*beta/N**2 + mu_b, 0],
[               I*beta/N - I*S*beta/N**2, -I*S*beta/N**2 - alpha - mu_d,       -I*S*beta/N**2 + S*beta/N,       -I*S*beta/N**2, 0],
[                                      0,                         alpha, -gamma*p - gamma*(1 - p) - mu_d,                    0, 0],
[                                      0,                             0,                   gamma*(1 - p),                -mu_d, 0],
[                                      0,                             0,                         gamma*p,                    0, 0]])

In [54]:
mod.ode_jacobian_states(state_example, time_example)

array([[-9.52380952e-03,  1.90750163e-02, -1.80924984e-01,
         1.90750163e-02,  0.00000000e+00],
       [ 9.52380952e-03, -5.19075016e-01,  1.80952381e-01,
        -1.90476190e-02,  0.00000000e+00],
       [ 0.00000000e+00,  5.00000000e-01, -2.50027397e-01,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  2.25000000e-01,
        -2.73972603e-05,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  2.50000000e-02,
         0.00000000e+00,  0.00000000e+00]])

### Event rates wrt params

In [55]:
mod.event_rates_jacobian_params.get_equation()

Matrix([
[I*S/N, 0,         0, 0, 0,        0],
[    0, E,         0, 0, 0,        0],
[    0, 0, I*(1 - p), 0, 0, -I*gamma],
[    0, 0,       I*p, 0, 0,  I*gamma],
[    0, 0,         0, N, 0,        0],
[    0, 0,         0, 0, S,        0],
[    0, 0,         0, 0, E,        0],
[    0, 0,         0, 0, I,        0],
[    0, 0,         0, 0, R,        0]])

In [56]:
mod.event_rates_jacobian_params(state_example, time_example)

array([[  66.66666667,    0.        ,    0.        ,    0.        ,
           0.        ,    0.        ],
       [   0.        ,  200.        ,    0.        ,    0.        ,
           0.        ,    0.        ],
       [   0.        ,    0.        ,   90.        ,    0.        ,
           0.        ,  -25.        ],
       [   0.        ,    0.        ,   10.        ,    0.        ,
           0.        ,   25.        ],
       [   0.        ,    0.        ,    0.        , 1050.        ,
           0.        ,    0.        ],
       [   0.        ,    0.        ,    0.        ,    0.        ,
         700.        ,    0.        ],
       [   0.        ,    0.        ,    0.        ,    0.        ,
         200.        ,    0.        ],
       [   0.        ,    0.        ,    0.        ,    0.        ,
         100.        ,    0.        ],
       [   0.        ,    0.        ,    0.        ,    0.        ,
          50.        ,    0.        ]])

### Event rates wrt states

In [57]:
mod.event_rates_jacobian_states.get_equation()

Matrix([
[I*beta/N - I*S*beta/N**2, -I*S*beta/N**2, -I*S*beta/N**2 + S*beta/N, -I*S*beta/N**2, 0],
[                       0,          alpha,                         0,              0, 0],
[                       0,              0,             gamma*(1 - p),              0, 0],
[                       0,              0,                   gamma*p,              0, 0],
[                    mu_b,           mu_b,                      mu_b,           mu_b, 0],
[                    mu_d,              0,                         0,              0, 0],
[                       0,           mu_d,                         0,              0, 0],
[                       0,              0,                      mu_d,              0, 0],
[                       0,              0,                         0,           mu_d, 0]])

In [58]:
mod.event_rates_jacobian_states(state_example, time_example)

array([[ 9.52380952e-03, -1.90476190e-02,  1.80952381e-01,
        -1.90476190e-02,  0.00000000e+00],
       [ 0.00000000e+00,  5.00000000e-01,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  2.25000000e-01,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  2.50000000e-02,
         0.00000000e+00,  0.00000000e+00],
       [ 2.73972603e-05,  2.73972603e-05,  2.73972603e-05,
         2.73972603e-05,  0.00000000e+00],
       [ 2.73972603e-05,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  2.73972603e-05,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  2.73972603e-05,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         2.73972603e-05,  0.00000000e+00]])

## Hessians

### ODE wrt params

In [59]:
mod.ode_hessian_params.get_equation()

[[[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, -I], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, -I, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, I], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, I, 0, 0, 0]]]

In [60]:
mod.ode_hessian_params(state_example, time_example)

array([[[   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.]],

       [[   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.]],

       [[   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.]],

       [[   0.,    0.,    0.,    0.,    0.,    0.],
      

### ODE wrt states

In [61]:
mod.ode_hessian_states.get_equation()

[[[2*I*beta/N**2 - 2*I*S*beta/N**3, I*beta/N**2 - 2*I*S*beta/N**3, I*beta/N**2 - 2*I*S*beta/N**3 - beta/N + S*beta/N**2, I*beta/N**2 - 2*I*S*beta/N**3, 0], [I*beta/N**2 - 2*I*S*beta/N**3, -2*I*S*beta/N**3, -2*I*S*beta/N**3 + S*beta/N**2, -2*I*S*beta/N**3, 0], [I*beta/N**2 - 2*I*S*beta/N**3 - beta/N + S*beta/N**2, -2*I*S*beta/N**3 + S*beta/N**2, -2*I*S*beta/N**3 + 2*S*beta/N**2, -2*I*S*beta/N**3 + S*beta/N**2, 0], [I*beta/N**2 - 2*I*S*beta/N**3, -2*I*S*beta/N**3, -2*I*S*beta/N**3 + S*beta/N**2, -2*I*S*beta/N**3, 0], [0, 0, 0, 0, 0]], [[-2*I*beta/N**2 + 2*I*S*beta/N**3, -I*beta/N**2 + 2*I*S*beta/N**3, -I*beta/N**2 + 2*I*S*beta/N**3 + beta/N - S*beta/N**2, -I*beta/N**2 + 2*I*S*beta/N**3, 0], [-I*beta/N**2 + 2*I*S*beta/N**3, 2*I*S*beta/N**3, 2*I*S*beta/N**3 - S*beta/N**2, 2*I*S*beta/N**3, 0], [-I*beta/N**2 + 2*I*S*beta/N**3 + beta/N - S*beta/N**2, 2*I*S*beta/N**3 - S*beta/N**2, 2*I*S*beta/N**3 - 2*S*beta/N**2, 2*I*S*beta/N**3 - S*beta/N**2, 0], [-I*beta/N**2 + 2*I*S*beta/N**3, 2*I*S*beta/N

In [62]:
mod.ode_hessian_states(state_example, time_example)

array([[[ 1.81405896e-05, -9.07029478e-06, -1.04308390e-04,
         -9.07029478e-06,  0.00000000e+00],
        [-9.07029478e-06, -3.62811791e-05,  1.54195011e-04,
         -3.62811791e-05,  0.00000000e+00],
        [-1.04308390e-04,  1.54195011e-04,  3.44671202e-04,
          1.54195011e-04,  0.00000000e+00],
        [-9.07029478e-06, -3.62811791e-05,  1.54195011e-04,
         -3.62811791e-05,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00]],

       [[-1.81405896e-05,  9.07029478e-06,  1.04308390e-04,
          9.07029478e-06,  0.00000000e+00],
        [ 9.07029478e-06,  3.62811791e-05, -1.54195011e-04,
          3.62811791e-05,  0.00000000e+00],
        [ 1.04308390e-04, -1.54195011e-04, -3.44671202e-04,
         -1.54195011e-04,  0.00000000e+00],
        [ 9.07029478e-06,  3.62811791e-05, -1.54195011e-04,
          3.62811791e-05,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
  

### ODE mixed wrt params+states

In [63]:
mod.ode_mixed_hessian_states_params.get_equation()

[[[-I/N + I*S/N**2, 0, 0, 1, -1, 0], [I*S/N**2, 0, 0, 1, 0, 0], [I*S/N**2 - S/N, 0, 0, 1, 0, 0], [I*S/N**2, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0]], [[I/N - I*S/N**2, 0, 0, 0, 0, 0], [-I*S/N**2, -1, 0, 0, -1, 0], [-I*S/N**2 + S/N, 0, 0, 0, 0, 0], [-I*S/N**2, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0], [0, 0, -1, 0, -1, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 1 - p, 0, 0, -gamma], [0, 0, 0, 0, -1, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, p, 0, 0, gamma], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]]]

In [64]:
mod.ode_mixed_hessian_states_params(state_example, time_example)

array([[[-0.03174603,  0.        ,  0.        ,  1.        ,
         -1.        ,  0.        ],
        [ 0.06349206,  0.        ,  0.        ,  1.        ,
          0.        ,  0.        ],
        [-0.6031746 ,  0.        ,  0.        ,  1.        ,
          0.        ,  0.        ],
        [ 0.06349206,  0.        ,  0.        ,  1.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ]],

       [[ 0.03174603,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [-0.06349206, -1.        ,  0.        ,  0.        ,
         -1.        ,  0.        ],
        [ 0.6031746 ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [-0.06349206,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ]],

       [[ 0.        ,  0. 

### Event rates wrt params

In [65]:
mod.event_rates_hessian_params.get_equation()

[[[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, -I], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, -I, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, I], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, I, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], 

In [66]:
mod.event_rates_hessian_params(state_example, time_example)

array([[[   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.]],

       [[   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.]],

       [[   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0., -100.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,    0., -100.,    0.,    0.,    0.]],

       [[   0.,    0.,    0.,    0.,    0.,    0.],
      

### Event rates wrt states

In [67]:
mod.event_rates_hessian_states.get_equation()

[[[-2*I*beta/N**2 + 2*I*S*beta/N**3, -I*beta/N**2 + 2*I*S*beta/N**3, -I*beta/N**2 + 2*I*S*beta/N**3 + beta/N - S*beta/N**2, -I*beta/N**2 + 2*I*S*beta/N**3, 0], [-I*beta/N**2 + 2*I*S*beta/N**3, 2*I*S*beta/N**3, 2*I*S*beta/N**3 - S*beta/N**2, 2*I*S*beta/N**3, 0], [-I*beta/N**2 + 2*I*S*beta/N**3 + beta/N - S*beta/N**2, 2*I*S*beta/N**3 - S*beta/N**2, 2*I*S*beta/N**3 - 2*S*beta/N**2, 2*I*S*beta/N**3 - S*beta/N**2, 0], [-I*beta/N**2 + 2*I*S*beta/N**3, 2*I*S*beta/N**3, 2*I*S*beta/N**3 - S*beta/N**2, 2*I*S*beta/N**3, 0], [0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0], [0, 0, 0, 

In [68]:
mod.event_rates_hessian_states(state_example, time_example)

array([[[-1.81405896e-05,  9.07029478e-06,  1.04308390e-04,
          9.07029478e-06,  0.00000000e+00],
        [ 9.07029478e-06,  3.62811791e-05, -1.54195011e-04,
          3.62811791e-05,  0.00000000e+00],
        [ 1.04308390e-04, -1.54195011e-04, -3.44671202e-04,
         -1.54195011e-04,  0.00000000e+00],
        [ 9.07029478e-06,  3.62811791e-05, -1.54195011e-04,
          3.62811791e-05,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00]],

       [[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
  

### Event rates mixed wrt params+states

In [69]:
mod.event_rates_mixed_hessian_states_params.get_equation()

[[[I/N - I*S/N**2, 0, 0, 0, 0, 0], [-I*S/N**2, 0, 0, 0, 0, 0], [-I*S/N**2 + S/N, 0, 0, 0, 0, 0], [-I*S/N**2, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 1 - p, 0, 0, -gamma], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, p, 0, 0, gamma], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 1, 0, 0], [0, 0, 0, 1, 0, 0], [0, 0, 0, 1, 0, 0], [0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0]]]

In [70]:
mod.event_rates_mixed_hessian_states_params(state_example, time_example)

array([[[ 0.03174603,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [-0.06349206,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.6031746 ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [-0.06349206,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ]],

       [[ 0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  1.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ]],

       [[ 0.        ,  0. 